In [ ]:
"""
extract_nasa_power.py

Purpose:
    Extract monthly NASA POWER data for every retained Philippine
    spatial grid cell using the representative query latitude
    and longitude generated during grid creation.

Variables:
    ALLSKY_SFC_SW_DWN = All Sky Surface Shortwave Downward Irradiance (kWh/m²/day)
    CLOUD_AMT         = Cloud Amount (%)
    RH2M              = Relative Humidity at 2 Meters (%)

Input:
    data/processed/grid_metadata.csv

Output:
    data/raw/nasa_power/nasa_power_monthly.csv
"""

from pathlib import Path
import time
import requests
import pandas as pd
from tqdm import tqdm


# ============================================================
# 1. USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# NASA POWER PARAMETERS
# ------------------------------------------------------------
# EDIT THIS SECTION if you want to add/remove variables.
#
# Current study variables:
#   ghi_kwh_m2_day -> ALLSKY_SFC_SW_DWN
#   cloud_amt_pct  -> CLOUD_AMT
#   rh2m_pct       -> RH2M
# ------------------------------------------------------------

NASA_PARAMETERS = (
    "ALLSKY_SFC_SW_DWN,"
    "CLOUD_AMT,"
    "RH2M"
)


# ------------------------------------------------------------
# NASA POWER COMMUNITY
# ------------------------------------------------------------

COMMUNITY = "RE"


# ------------------------------------------------------------
# STUDY PERIOD
# ------------------------------------------------------------

START_YEAR = 2005
END_YEAR = 2025


# ------------------------------------------------------------
# INPUT FILE
# ------------------------------------------------------------

GRID_FILE = Path(
    "data/processed/grid_metadata.csv"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "data/raw/nasa_power"
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "nasa_power_monthly.csv"
)


# ------------------------------------------------------------
# NASA POWER API
# ------------------------------------------------------------

BASE_URL = (
    "https://power.larc.nasa.gov/api/temporal/monthly/point"
)


# ------------------------------------------------------------
# REQUEST SETTINGS
# ------------------------------------------------------------

REQUEST_TIMEOUT = 60

# Small delay between API requests
REQUEST_DELAY = 0.2


# ============================================================
# 2. CREATE OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 3. LOAD GRID METADATA
# ============================================================

print("\nLoading grid metadata...")

if not GRID_FILE.exists():

    raise FileNotFoundError(
        f"""
Grid metadata file was not found:

{GRID_FILE}

Please run create_grid.py first.
"""
    )


grid = pd.read_csv(GRID_FILE)

print(
    f"Grid cells found: {len(grid)}"
)


# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "grid_number",
    "grid_id",
    "query_latitude",
    "query_longitude"
]


missing_columns = [
    column
    for column in required_columns
    if column not in grid.columns
]


if missing_columns:

    raise ValueError(
        f"""
The following required columns are missing
from grid_metadata.csv:

{missing_columns}

Make sure your grid-generation script contains:
    grid_number
    grid_id
    query_latitude
    query_longitude
"""
    )


# ============================================================
# 5. NASA POWER API FUNCTION
# ============================================================

def get_nasa_power_data(
    latitude,
    longitude,
    start_year,
    end_year
):
    """
    Request monthly NASA POWER data for one
    geographic coordinate.
    """

    params = {

        "parameters": NASA_PARAMETERS,

        "community": COMMUNITY,

        "longitude": longitude,

        "latitude": latitude,

        "start": start_year,

        "end": end_year,

        "format": "JSON"
    }


    response = requests.get(
        BASE_URL,
        params=params,
        timeout=REQUEST_TIMEOUT
    )


    response.raise_for_status()


    return response.json()


# ============================================================
# 6. PROCESS ONE GRID CELL
# ============================================================

def process_grid_cell(row):

    grid_number = row["grid_number"]

    grid_id = row["grid_id"]

    latitude = row["query_latitude"]

    longitude = row["query_longitude"]


    try:

        # ----------------------------------------------------
        # Request NASA POWER
        # ----------------------------------------------------

        data = get_nasa_power_data(
            latitude=latitude,
            longitude=longitude,
            start_year=START_YEAR,
            end_year=END_YEAR
        )


        # ----------------------------------------------------
        # Retrieve parameter dictionary
        # ----------------------------------------------------

        parameters = (
            data["properties"]["parameter"]
        )


        # ----------------------------------------------------
        # Retrieve individual variables
        # ----------------------------------------------------

        ghi_data = parameters[
            "ALLSKY_SFC_SW_DWN"
        ]

        cloud_data = parameters[
            "CLOUD_AMT"
        ]

        rh2m_data = parameters[
            "RH2M"
        ]


        # ----------------------------------------------------
        # Convert monthly data into records
        # ----------------------------------------------------

        records = []


        for date_key in ghi_data.keys():

            year = int(
                date_key[:4]
            )

            month = int(
                date_key[4:]
            )


            records.append({

                "grid_number":
                    grid_number,

                "grid_id":
                    grid_id,

                "latitude":
                    latitude,

                "longitude":
                    longitude,

                "year":
                    year,

                "month":
                    month,

                "date":
                    pd.to_datetime(
                        f"{year}-{month:02d}-01"
                    ),

                "ghi_kwh_m2_day":
                    ghi_data[date_key],

                "cloud_amt_pct":
                    cloud_data.get(
                        date_key
                    ),

                "rh2m_pct":
                    rh2m_data.get(
                        date_key
                    )
            })


        return records


    except Exception as error:

        print(
            f"""
ERROR
Grid number: {grid_number}
Grid ID:     {grid_id}
Latitude:    {latitude}
Longitude:   {longitude}

Error:
{error}
"""
        )

        return []


# ============================================================
# 7. START EXTRACTION
# ============================================================

print("\n")
print("=" * 65)
print("NASA POWER DATA EXTRACTION")
print("=" * 65)

print(
    f"Parameters: {NASA_PARAMETERS}"
)

print(
    f"Period:     {START_YEAR}-{END_YEAR}"
)

print(
    f"Grid cells: {len(grid)}"
)

print("=" * 65)
print()


# ============================================================
# 8. EXTRACT ALL GRID CELLS
# ============================================================

all_records = []


for _, row in tqdm(
    grid.iterrows(),
    total=len(grid),
    desc="Downloading NASA POWER"
):

    records = process_grid_cell(row)

    all_records.extend(
        records
    )

    time.sleep(
        REQUEST_DELAY
    )


# ============================================================
# 9. CHECK EXTRACTION
# ============================================================

if not all_records:

    raise RuntimeError(
        """
No NASA POWER data were successfully retrieved.

Check:
1. Internet connection
2. NASA POWER API availability
3. Grid coordinates
4. Parameter names
5. Study period
"""
    )


# ============================================================
# 10. CREATE DATAFRAME
# ============================================================

df = pd.DataFrame(
    all_records
)


# ============================================================
# 11. SORT DATA
# ============================================================

df = df.sort_values(
    by=[
        "grid_number",
        "date"
    ]
).reset_index(
    drop=True
)


# ============================================================
# 12. CHECK EXPECTED OBSERVATIONS
# ============================================================

expected_per_grid = (
    (END_YEAR - START_YEAR + 1)
    * 12
)


expected_total = (
    len(grid)
    * expected_per_grid
)


print("\n")
print("=" * 65)
print("DATA VALIDATION")
print("=" * 65)

print(
    f"Expected observations per grid: "
    f"{expected_per_grid}"
)

print(
    f"Expected total observations: "
    f"{expected_total}"
)

print(
    f"Actual total observations: "
    f"{len(df)}"
)


# ============================================================
# 13. CHECK GRID COMPLETENESS
# ============================================================

grid_counts = (
    df.groupby("grid_number")
    .size()
)


incomplete_grids = (
    grid_counts[
        grid_counts != expected_per_grid
    ]
)


if len(incomplete_grids) > 0:

    print("\nWARNING:")
    print(
        f"{len(incomplete_grids)} grid cells "
        "do not contain the expected number "
        "of monthly observations."
    )

    print(
        incomplete_grids
    )

else:

    print(
        "\nAll grid cells contain the "
        "expected number of observations."
    )


# ============================================================
# 14. CHECK MISSING VALUES
# ============================================================

print("\nMissing values:")

print(
    df[
        [
            "ghi_kwh_m2_day",
            "cloud_amt_pct",
            "rh2m_pct"
        ]
    ]
    .isna()
    .sum()
)


# ============================================================
# 15. SAVE DATASET
# ============================================================

df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 65)
print("NASA POWER EXTRACTION COMPLETE")
print("=" * 65)

print(
    f"Grid cells:       "
    f"{df['grid_number'].nunique()}"
)

print(
    f"Date range:       "
    f"{df['date'].min().date()} "
    f"to "
    f"{df['date'].max().date()}"
)

print(
    f"Total records:    "
    f"{len(df)}"
)

print(
    f"Expected records: "
    f"{expected_total}"
)

print(
    f"\nOutput file:\n"
    f"{OUTPUT_FILE}"
)


# ============================================================
# 17. DISPLAY SAMPLE
# ============================================================

print("\nFirst 10 records:")

print(
    df.head(10).to_string(
        index=False
    )
)


print("\n")
print("Done.")